# Midterm - Business-Case Predictive Strategy Practicum + Project Baseline Submission

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb10_midterm_casebook_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Translate business cases into predictive tasks (target, unit, horizon, KPI)
2. Select split strategy and metrics aligned to case and cost structure
3. Identify leakage risks and data availability constraints
4. Propose a modeling shortlist and an evaluation plan
5. Deliver a baseline model + evaluation plan for the course project

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## Midterm Instructions

Take a breath before you dive in. This is a **midterm**, but it is not a trick. You have done everything the two cases ask in nb01 through nb09 — today is the exercise of picking the right tool for a new problem statement and defending the choice.

### What this midterm actually tests

The midterm is a **design exercise, not a coding exercise**. You will not be graded on whether your code runs, or on how clever the code is. You will be graded on four things, the same four things every modeling project at any company is graded on:

1. **Did you define the target cleanly?** One row = one what? One column = one why? If a stakeholder asked "what are we predicting, in plain English," could you answer in one sentence?
2. **Did you choose a metric that aligns with the business cost structure?** Accuracy is rarely the right answer. The case prompts give you explicit dollar costs — translate them.
3. **Did you design a split strategy that prevents leakage?** The same stratify-and-lock rules from nb01 apply here.
4. **Did you pick a sensible baseline and a path from baseline to improvement?** "Logistic regression, then tree, then gradient boosting" is a defensible path. "Let's try XGBoost" with no baseline is not.

### Academic Integrity

**Allowed resources:**
- Course notebooks (Days 1–9) — **absolutely use them**; this is a closed-book world you are simulating, not a closed-resource one
- scikit-learn documentation
- Course textbooks (ISLP, ESL, Provost & Fawcett)
- Gemini for code generation and concept explanation (with the Ask → Verify → Document workflow from nb00)

**Not allowed:**
- Communicating with classmates or other test-takers during the exam
- Pre-written code from external sources without attribution
- Asking Gemini to solve entire case problems for you

**Gemini usage boundaries** — the same pattern you have used all semester:
- ✓ Ask Gemini to generate a code scaffold ("write a pipeline with a scaler and a logistic regression")
- ✓ Ask Gemini to explain a concept you forgot ("remind me what `stratify=y` does and why it matters here")
- ✓ Use Gemini to debug an error you already understand the shape of
- ✗ Paste the entire case prompt and say "solve this"
- ✗ Copy Gemini's analysis into your response without reading it critically

> **One meta-note:** the cheat sheet at the end of this notebook (Section "Midterm Cheat Sheet") compresses every decision table from nb01–nb09 into one page. **Scroll down and open it before you start Case 1** — it is designed to live on your screen while you work.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The setup cell brings in the libraries you *might* need during the midterm — `pandas` and `numpy` for data manipulation, `matplotlib` and `seaborn` for visualization, and the full scikit-learn toolkit (`train_test_split`, `StratifiedKFold`, `Pipeline`, `StandardScaler`, `LogisticRegression`, `classification_report`, `roc_auc_score`). `RANDOM_SEED = 474` is locked for reproducibility, consistent with nb01–nb09.

The "**Setup complete!**" confirmation means your environment is ready. You will not need to install any additional packages.

**An important framing point:** the midterm is a design exercise. You will not be asked to fit a model from scratch. The setup cell is here in case you choose to write a quick code snippet to illustrate an idea (for example, computing an imbalanced-class stratification check, or sketching a CV loop to make a point about split strategy) — but you can earn full credit on this midterm **without writing a single line of code**. The grading rubric rewards strategic reasoning: target definition, metric alignment, leakage prevention, and modeling path. The imports are a safety net, not a requirement.

> **A question that often comes up here:** *"If I do write code to support my answer, does it need to execute cleanly?"* Yes — if you include a cell that runs, it should run without error. But write code only if it makes your argument *clearer*. Many strong midterm responses are entirely prose + tables.

---

## Case 1: Customer Churn Prediction for Subscription Service

### Business Context

**Company:** StreamFlix, a video streaming service.

**Problem:** Monthly churn rate is 5% — one in every twenty customers cancels each month. Acquiring a new customer costs \$50 (marketing + onboarding); retaining an existing customer who was about to leave costs \$10 (a time-limited discount offer). The product team wants a targeted retention campaign that reaches the right 5% before they cancel.

**Data available:**
- Customer demographics (age, location, device type)
- Usage patterns (watch hours, genres watched, last-login recency)
- Account history (tenure, payment method, support ticket count)
- Churn outcome (binary — did the customer cancel in the following month, yes or no)

**Business goal:** identify customers likely to churn in the next 30 days so the retention team can offer them a discount before they leave.

Before you start writing — **walk through the case in your head for 30 seconds first.** The cheapest way to fail this case is to miss a mundane detail: who is in the dataset, what does one row mean, when was the label generated relative to the features. Those three questions determine the split strategy, the metric, and the leakage risks.

### Your Task (Case 1)

Design a predictive modeling plan that addresses **all five** of the following. Write your response in the cell labeled `### CASE 1: YOUR RESPONSE` further down — not here. The bullets below are the prompt; your response goes in the response cell.

1. **Prediction Target** — what exactly are you predicting, in one sentence?
2. **Prediction Unit** — what does one row in the dataset represent? (One customer? One customer-month? This matters for leakage.)
3. **Prediction Horizon** — how far in the future is the label? (Next 7 days? Next 30? This matters for split strategy.)
4. **Primary Metric** — which metric aligns with the \$50 / \$10 cost structure? Justify.
5. **Modeling Shortlist** — what three models, in what order, would you try? Justify the order.

> **A question that often comes up during this case:** *"Should I worry about time ordering in the split?"* Yes — and this is one of the most common places to lose credit. The label is "churned in the next 30 days." That means the split cannot be random across rows; it has to respect time. Use the most recent 20% of observations for validation and the next-most-recent 20% for test (or use a time-aware splitter like `TimeSeriesSplit`). A random split would let the model peek at the future, and the CV score would be inflated for exactly that reason.

---

## 📝 PAUSE-AND-DO: Case 1 Response (5 minutes)

Write your structured response below.

---

### CASE 1: YOUR RESPONSE

#### 1. Prediction Target
[What exactly are we predicting? Be specific.]

**Your answer:**

---

#### 2. Prediction Unit
[What is one row? Customer-month? Customer? Subscription?]

**Your answer:**

---

#### 3. Prediction Horizon
[How far ahead? Why this timeframe?]

**Your answer:**

---

#### 4. Primary Metric
[Which metric and why? Consider business costs.]

**Your answer:**

**Justification based on costs:**
- Cost of False Negative (missing a churner): $\_\_\_\_
- Cost of False Positive (unnecessary retention offer): $\_\_\_\_
- Therefore, metric should be: \_\_\_\_\_

---

#### 5. Split Strategy
[Random? Time-based? Stratified? Why?]

**Your answer:**

---

#### 6. Leakage Risks
[List 3 specific features or patterns that could cause leakage]

**Risk 1:**

**Risk 2:**

**Risk 3:**

---

#### 7. Model Shortlist
[Which 2-3 models would you try? Why?]

**Model 1:**

**Model 2:**

**Model 3 (optional):**

---

#### 8. Threshold Selection
[How would you choose the threshold? What factors matter?]

**Your answer:**

---

## Case 2: Loan Default Prediction

### Business Context

**Company:** A FinTech lending platform.

**Problem:** Approve loans for creditworthy applicants and reject risky ones. Each decision is expensive to get wrong in both directions.

**Data available:**
- Applicant demographics (age, income, employment status)
- Credit history (credit score, number of previous loans, payment history)
- Loan details (requested amount, stated purpose, term length)
- Default outcome (binary — did the applicant default within 12 months, yes or no)

**Business constraints — note the asymmetry:**
- Average loan amount: \$10,000
- Average profit per successful (paid-back) loan: \$500
- Average loss per default: \$7,000
- Regulatory requirement: maintain a default rate below **3%** at all times

### Your Task (Case 2)

This case is the calibration-to-action case — the hard thing is translating the \$7,000-loss-vs-\$500-profit asymmetry into an evaluation plan, and translating the 3% regulatory ceiling into a decision threshold. Write your response in `### CASE 2: YOUR RESPONSE`.

1. **Primary Metric** — what metric aligns with the profit/loss structure? (Hint: think about expected value of a decision, not about accuracy.)
2. **Constraint Handling** — how do you enforce the 3% regulatory default-rate ceiling? (Hint: this is a threshold question, not a model question.)
3. **Class Imbalance** — default rates are typically 2–5% of applicants. What are your two or three first-resort tools for handling that, and in what order?
4. **Split Strategy** — time-aware or random? Justify.
5. **Risk of Leakage** — name **two** concrete leakage risks specific to this dataset (not generic). Explain how you would detect each.

> **A question that often comes up during this case:** *"The regulatory 3% ceiling feels like a modeling constraint — but is it?"* It is actually a *threshold* constraint. You set your decision threshold so that among the loans you approve, no more than 3% default — regardless of which model is underneath. That decouples model selection (maximize expected value) from the operational ceiling (set the threshold high enough to meet regulation). If you write that sentence clearly in your response, you are already earning most of the constraint-handling credit on this case.

---

## 📝 PAUSE-AND-DO: Case 2 Response (5 minutes)

---

### CASE 2: YOUR RESPONSE

#### 1. Primary Metric
[What metric captures profit/loss tradeoff?]

**Your answer:**

**Expected value calculation:**
- E[value | approve good applicant] = $\_\_\_\_
- E[value | approve bad applicant] = $\_\_\_\_
- E[value | reject good applicant] = $\_\_\_\_
- E[value | reject bad applicant] = $\_\_\_\_

---

#### 2. Constraint Handling
[How to enforce default rate < 3%?]

**Your answer:**

---

#### 3. Class Imbalance Handling
[Defaults are 1-2%. What strategies would you use?]

**Strategy 1:**

**Strategy 2:**

**Strategy 3:**

---

#### 4. Decision Rule
[Write the logic for loan approval]

**Your answer:**

```
if probability_default < threshold:
    approve loan
else:
    reject loan

threshold = ???? (explain how you'd set this)
```

---

#### 5. Production Monitoring
[What metrics would you track over time?]

**Metric 1:**

**Metric 2:**

**Metric 3:**

**Metric 4:**

---

## Mini-Case 3: Medical Diagnosis (Optional Bonus)

### Quick Scenario

**Task:** Predict rare disease (0.1% prevalence) from lab tests

**Constraints:**
- False negative (missing disease) = life-threatening
- False positive = unnecessary expensive test (\$5,000)
- Dataset: 100,000 patients, 100 with disease

**Questions:**
1. Which metric: Precision, Recall, F1, ROC-AUC, or PR-AUC? Why?
2. What threshold bias (high/low)? Why?
3. How to handle 100:99,900 class imbalance?

---

### MINI-CASE 3: YOUR RESPONSE (OPTIONAL)

**Metric choice:**

**Threshold bias:**

**Imbalance handling:**

---

## Project Milestone 2: Baseline Model + Evaluation Plan

Your project baseline submission is due today. This is a working deliverable, not a final model. A strong baseline looks *plain* — a single pipeline, a single metric, a single plot — and leaves lots of room for Week 3 and Week 4 to improve it. A weak baseline over-engineers before the evaluation plan is nailed down.

### Deliverable Requirements

Your submission must include:

#### 1. Baseline Pipeline (code, not prose)
- Train / val / test splits with `RANDOM_SEED = 474` and appropriate stratification
- Preprocessing inside a `Pipeline` (or `ColumnTransformer` if you have categorical columns — today you do)
- At least one baseline model (logistic regression, ridge regression, or equivalent)
- Evaluation on validation set

#### 2. Baseline Report (table)
- Baseline model
- At least one slightly-improved model (regularized, different C or α, etc.)
- 2–3 supporting metrics alongside the primary metric
- Clear identification of which one is the **current** champion (it will change by Week 3)

#### 3. Evaluation Plan (short prose block)
- Primary metric + justification ("we use ROC-AUC because the stakeholder cares about ranking, not about a specific threshold — the threshold gets picked in nb16")
- Split / CV design
- Leakage-prevention measures (explicit — name the patterns from nb09 you avoided)
- Next steps ("tree-based models, then ensembles, then calibration")

> **A question that often comes up here:** *"My baseline score is not impressive. Should I try to boost it before submitting?"* No. The point of the baseline is to set a **reference floor** — the number Week 3 models have to beat by a CI-clearing margin to earn their complexity. A baseline that already scores near the ceiling leaves no room for improvement, which defeats the pedagogical arc of the project. If your baseline ROC-AUC is 0.70, that is a *good* number for a baseline. If it is 0.97, check whether you have a feature that is too-close-to-the-label (a common mistake), because a baseline that high usually signals leakage.

---

## Grading Rubric

### Midterm Cases (70 points)

**Case 1 (35 points)**
- Problem framing (10 pts): Target, unit, horizon clearly defined
- Metric selection (10 pts): Justified by business costs
- Leakage awareness (5 pts): Specific, realistic risks identified
- Modeling plan (10 pts): Reasonable shortlist + threshold strategy

**Case 2 (35 points)**
- Expected value logic (10 pts): Correct cost calculations
- Constraint handling (10 pts): Concrete approach to 3% limit
- Imbalance handling (10 pts): Multiple actionable strategies
- Monitoring plan (5 pts): Relevant production metrics

### Project Baseline (30 points)
- Code quality (10 pts): Runs without errors, proper pipeline
- Evaluation rigor (10 pts): Appropriate metrics, no leakage
- Documentation (10 pts): Clear plan, justified choices

---

## Common Mistakes to Avoid

### Case Analysis Mistakes
- ✗ Generic answers ("use cross-validation")
- ✓ Specific answers ("use time-based split because...")

- ✗ Ignoring business costs
- ✓ Justify metrics with cost structure

- ✗ Vague leakage risks ("data leakage might happen")
- ✓ Specific risks ("including 'account_status_after_churn' would leak")

### Code Mistakes
- ✗ Fitting on full dataset before split
- ✓ Split first, then fit only on train

- ✗ Looking at test set during development
- ✓ Lock test set, use only validation

- ✗ No baseline for comparison
- ✓ Always include simple baseline

---

## How to Earn Full Credit

### Excellent Response Characteristics

1. **Specificity**: Answers are tailored to the specific case
2. **Justification**: Every choice is explained with reasoning
3. **Business alignment**: Technical choices driven by business needs
4. **Completeness**: All parts of each question addressed
5. **Realistic**: Acknowledges tradeoffs and limitations

### Example of Excellent vs Poor Answer

**Question:** Which metric for churn prediction?

**Poor answer:**  
"I would use accuracy because it's a good metric."

**Excellent answer:**  
"I would use PR-AUC as primary metric with a focus on recall at 20% precision. Reasoning: (1) Churn is likely imbalanced (\~5% rate), making accuracy misleading. (2) Missing churners (FN) costs \$50 (lost customer acquisition cost), while unnecessary retention offers (FP) cost only \$10. (3) We can afford high FP rate if it means catching most churners. (4) Will use precision-recall curve to find optimal operating point where cost is minimized, likely favoring high recall even with moderate precision."

---

## 📋 Midterm Cheat Sheet — Decision Tables (open during the midterm)

Copy this into a new cell or a separate window while you work the cases.

### Which metric for which problem?

| Problem type | Primary metric | When to also report |
|---|---|---|
| Regression, predictions in native units matter | MAE | RMSE for severity of outliers; $R^2$ for variance explained |
| Regression, asymmetric error costs | Custom cost or quantile loss | RMSE as baseline reference |
| Binary classification, balanced classes | ROC-AUC | Accuracy, F1 |
| Binary classification, imbalanced classes | PR-AUC or ROC-AUC + threshold tuning | Recall on minority class, precision on minority class |
| Probabilistic decisions (screening, pricing) | Brier score or log-loss | ROC-AUC as discrimination check (nb16) |

### Scaler choice

| Model family | Scaler | Why |
|---|---|---|
| Logistic / Ridge / Lasso / Linear Regression | `StandardScaler` inside `Pipeline` | Gradient-based optimizers and L1/L2 penalties require comparable feature scales |
| Decision tree, Random Forest, Gradient Boosting | **No scaler** | Trees use thresholds, not distances — scale-invariant |
| KNN, SVM with RBF | `StandardScaler` | Distance-based methods are highly scale-sensitive |

### Stratify or not?

| Split scenario | `stratify=` |
|---|---|
| Classification, any class balance | `stratify=y` — always |
| Regression | Not needed (default) |
| Classification, time-ordered data | Neither — use `TimeSeriesSplit` instead |

### Ridge vs. Lasso

| Use Ridge when… | Use Lasso when… |
|---|---|
| All features plausibly matter | Feature selection is a deliverable |
| Features are correlated (Lasso is unstable across correlated features) | Features are roughly uncorrelated |
| Goal is prediction | Goal is interpretation + prediction |
| A small, dense coefficient vector is fine | A sparse coefficient vector is required (regulatory, communication) |

### CI-overlap rule (from nb08/nb09)

Compare two CV estimates with 5 fold scores each:

1. Compute each mean and each 95% CI: `ci = t.ppf(0.975, df=4) * sd / sqrt(5)` → `[mean - ci, mean + ci]`.
2. If the two CIs **overlap**, the two models are statistically indistinguishable on this data. Pick the simpler one.
3. If the two CIs **do not overlap**, the higher-mean model is the defensible winner.

### Leakage checklist (from nb09)

- [ ] Every `.fit` call on data-derived transforms happens **inside** the `Pipeline` the CV sees.
- [ ] No `SelectKBest`, `StandardScaler`, `SimpleImputer`, or target encoder fit before the train/test split.
- [ ] No feature derived from `y` on any row that will land in a held-out fold.
- [ ] Categorical encoding uses `OneHotEncoder(handle_unknown='ignore')`, not a manual `get_dummies` on full data.

Keep this cheat sheet open. Every question in both cases can be answered by picking the right row from one of these tables.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- Provost, F., & Fawcett, T. (2013). *Data Science for Business* - End-to-end predictive modeling process and business framing
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Assessment/selection + classification/regression chapters
- scikit-learn User Guide: [Common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html) - Especially leakage and improper evaluation

---



<center>

Thank you!

</center>